In [0]:
print("Hello from Databricks")

Hello from Databricks


In [0]:
spark

In [0]:
documents = [
    {
        "document_name": "troubleshooting_guide.md",
        "section": "Slow Internet",
        "content": "Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support.",
        "source": "troubleshooting"
    },
    {
        "document_name": "billing_policy.md",
        "section": "Billing Disputes",
        "content": "Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.",
        "source": "billing"
    },
    {
        "document_name": "refund_policy.md",
        "section": "Refund Eligibility",
        "content": "Customers may request a refund within 30 days of activation if the service was unavailable for more than 72 continuous hours due to a provider-side issue.",
        "source": "refunds"
    }
]

In [0]:
df = spark.createDataFrame(documents)
display(df)

content,document_name,section,source
"Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support.",troubleshooting_guide.md,Slow Internet,troubleshooting
Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.,billing_policy.md,Billing Disputes,billing
Customers may request a refund within 30 days of activation if the service was unavailable for more than 72 continuous hours due to a provider-side issue.,refund_policy.md,Refund Eligibility,refunds


In [0]:
df.printSchema()

root
 |-- content: string (nullable = true)
 |-- document_name: string (nullable = true)
 |-- section: string (nullable = true)
 |-- source: string (nullable = true)



In [0]:
from pyspark.sql.functions import current_timestamp, length, col

df_enriched = (
    df
    .withColumn("content_length", length(col("content")))
    .withColumn("loaded_at", current_timestamp())
)

display(df_enriched)

content,document_name,section,source,content_length,loaded_at
"Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support.",troubleshooting_guide.md,Slow Internet,troubleshooting,187,2026-05-23T05:47:49.738Z
Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.,billing_policy.md,Billing Disputes,billing,158,2026-05-23T05:47:49.738Z
Customers may request a refund within 30 days of activation if the service was unavailable for more than 72 continuous hours due to a provider-side issue.,refund_policy.md,Refund Eligibility,refunds,154,2026-05-23T05:47:49.738Z


In [0]:
df_enriched.write.format("delta").mode("overwrite").saveAsTable("acmenet_bronze_documents")

In [0]:
bronze_df = spark.table("acmenet_bronze_documents")
display(bronze_df)

content,document_name,section,source,content_length,loaded_at
"Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support.",troubleshooting_guide.md,Slow Internet,troubleshooting,187,2026-05-23T05:50:11.972Z
Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.,billing_policy.md,Billing Disputes,billing,158,2026-05-23T05:50:11.972Z
Customers may request a refund within 30 days of activation if the service was unavailable for more than 72 continuous hours due to a provider-side issue.,refund_policy.md,Refund Eligibility,refunds,154,2026-05-23T05:50:11.972Z


In [0]:
%sql

SELECT 
  source,
  COUNT(*) AS total_documents,
  AVG(content_length) AS avg_content_length
FROM acmenet_bronze_documents
GROUP BY source
ORDER BY total_documents DESC;

source,total_documents,avg_content_length
troubleshooting,1,187.0
billing,1,158.0
refunds,1,154.0
